In [1]:
import numpy as np
np.int = int
np.float = float

import pandas as pd
import geoplotlib
from geoplotlib.utils import DataAccessObject

In [2]:
from pathlib import Path

paths = [Path('world_cities_pop.csv'), Path('../Datasets/world_cities_pop.csv')]
file_path = next((p for p in paths if p.exists()), None)
if file_path is None:
    raise FileNotFoundError('Place world_cities_pop.csv beside this notebook or in ../Datasets/')

dataset = pd.read_csv(file_path, dtype={'Region': str}, low_memory=False)
dataset.head()

,Country,City,AccentCity,Region,Population,Latitude,Longitude
0,ad,aixas,Aixàs,06,NaN,42.483333,1.466667
1,ad,aixirivali,Aixirivali,06,NaN,42.466667,1.500000
2,ad,aixirivall,Aixirivall,06,NaN,42.466667,1.500000
3,ad,aixirvall,Aixirvall,06,NaN,42.466667,1.500000
4,ad,aixovall,Aixovall,06,NaN,42.466667,1.483333


In [3]:
dataset.info()
print(dataset.dtypes)
print(dataset.shape)
print(dataset.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 3173958 entries, 0 to 3173957
Data columns (total 7 columns):
 #   Column      Dtype  
---  ------      -----  
 0   Country     str    
 1   City        str    
 2   AccentCity  str    
 3   Region      str    
 4   Population  float64
 5   Latitude    float64
 6   Longitude   float64
dtypes: float64(3), str(4)
memory usage: 169.5 MB
Country           str
City              str
AccentCity        str
Region            str
Population    float64
Latitude      float64
Longitude     float64
dtype: object
(3173958, 7)
['Country', 'City', 'AccentCity', 'Region', 'Population', 'Latitude', 'Longitude']


In [4]:
dataset['lat'] = pd.to_numeric(dataset['Latitude'], errors='coerce')
dataset['lon'] = pd.to_numeric(dataset['Longitude'], errors='coerce')
dataset['Population'] = pd.to_numeric(dataset['Population'], errors='coerce')
dataset['Country'] = dataset['Country'].astype('string').str.lower().str.strip()

dataset = dataset.dropna(subset=['lat', 'lon'])
dataset = dataset[
    dataset['lat'].between(-90, 90) & dataset['lon'].between(-180, 180)
].copy()
dataset[['Country', 'Population', 'lat', 'lon']].head()

,Country,Population,lat,lon
0,ad,NaN,42.483333,1.466667
1,ad,NaN,42.466667,1.500000
2,ad,NaN,42.466667,1.500000
3,ad,NaN,42.466667,1.500000
4,ad,NaN,42.466667,1.483333


In [5]:
def geo_data(frame):
    return DataAccessObject({
        'lat': frame['lat'].to_numpy(dtype=float),
        'lon': frame['lon'].to_numpy(dtype=float)
    })

geoplotlib.dot(geo_data(dataset))
geoplotlib.show()

In [6]:
cities_per_country = (
    dataset.groupby('Country')
    .agg(city_count=('Country', 'size'))
    .reset_index()
)

average_cities = cities_per_country['city_count'].mean()
print('Average number of cities per country:', average_cities)
cities_per_country.head(20)

Average number of cities per country: 13563.923076923076


,Country,city_count
0,ad,92
1,ae,446
2,af,88749
3,ag,183
4,ai,42
5,al,15123
6,am,2890
7,an,269
8,ao,19560
9,ar,8738


In [7]:
dataset_positive = dataset[dataset['Population'] > 0].copy()
print('Number of cities with population > 0:', len(dataset_positive))
dataset_positive.head(20)

Number of cities with population > 0: 47980


,Country,City,AccentCity,Region,Population,Latitude,Longitude,lat,lon
6,ad,andorra la vella,Andorra la Vella,07,20430.0,42.500000,1.516667,42.500000,1.516667
20,ad,canillo,Canillo,02,3292.0,42.566667,1.600000,42.566667,1.600000
32,ad,encamp,Encamp,03,11224.0,42.533333,1.583333,42.533333,1.583333
49,ad,la massana,La Massana,04,7211.0,42.550000,1.516667,42.550000,1.516667
53,ad,les escaldes,Les Escaldes,08,15854.0,42.500000,1.533333,42.500000,1.533333
66,ad,ordino,Ordino,05,2553.0,42.550000,1.533333,42.550000,1.533333
80,ad,sant julia de loria,Sant Julià de Lòria,06,8020.0,42.466667,1.500000,42.466667,1.500000
93,ae,abu dhabi,Abu Dhabi,01,603687.0,24.466667,54.366667,24.466667,54.366667
242,ae,dubai,Dubai,03,1137376.0,25.258172,55.304717,25.258172,55.304717
490,ae,sharjah,Sharjah,06,543942.0,25.357310,55.403304,25.357310,55.403304


In [8]:
geoplotlib.dot(geo_data(dataset_positive))
geoplotlib.show()

In [9]:
dataset_large = dataset_positive[dataset_positive['Population'] > 100000].copy()
print('Number of cities with population > 100,000:', len(dataset_large))
dataset_large.head()

Number of cities with population > 100,000: 3527


,Country,City,AccentCity,Region,Population,Latitude,Longitude,lat,lon
93,ae,abu dhabi,Abu Dhabi,01,603687.0,24.466667,54.366667,24.466667,54.366667
242,ae,dubai,Dubai,03,1137376.0,25.258172,55.304717,25.258172,55.304717
490,ae,sharjah,Sharjah,06,543942.0,25.357310,55.403304,25.357310,55.403304
6644,af,baglan,Baglan,03,108481.0,36.130684,68.708286,36.130684,68.708286
24457,af,gardez,Gardez,36,103732.0,33.597439,69.225922,33.597439,69.225922


In [10]:
geoplotlib.voronoi(geo_data(dataset_large), cmap='hot_r')
geoplotlib.show()

In [11]:
dataset_germany_uk = dataset_large[
    dataset_large['Country'].isin(['de', 'gb', 'uk'])
].copy()
print(dataset_germany_uk['Country'].value_counts())
print('Selected cities:', len(dataset_germany_uk))
dataset_germany_uk.head()

Country
de    81
gb    69
Name: count, dtype: Int64
Selected cities: 150


,Country,City,AccentCity,Region,Population,Latitude,Longitude,lat,lon
722416,de,aachen,Aachen,07,251104.0,50.770833,6.105278,50.770833,6.105278
725123,de,augsburg,Augsburg,02,261842.0,48.366667,10.883333,48.366667,10.883333
727047,de,bergisch gladbach,Bergisch Gladbach,07,106611.0,50.983333,7.133333,50.983333,7.133333
727152,de,berlin,Berlin,16,3398362.0,52.516667,13.400000,52.516667,13.400000
727693,de,bielefeld,Bielefeld,07,327198.0,52.033333,8.533333,52.033333,8.533333


In [12]:
geoplotlib.delaunay(geo_data(dataset_germany_uk), cmap='hot_r')
geoplotlib.show()

Traceback (most recent call last):
  File "D:\geoplotlib_env\Lib\site-packages\geoplotlib\__init__.py", line 32, in _runapp
    app.start()
  File "D:\geoplotlib_env\Lib\site-packages\geoplotlib\core.py", line 367, in start
    l.invalidate(self.proj)
  File "D:\geoplotlib_env\Lib\site-packages\geoplotlib\layers.py", line 433, in invalidate
    for tria in dela.vertices:
                ^^^^^^^^^^^^^
AttributeError: 'Delaunay' object has no attribute 'vertices'


In [13]:
geoplotlib.dot(geo_data(dataset_germany_uk))
geoplotlib.show()

In [14]:
import sys
import numpy as np

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Dataset size:", dataset.shape)

Python: D:\geoplotlib_env\Scripts\python.exe
NumPy: 1.26.4
Dataset size: (3173958, 9)


In [15]:
import geoplotlib
from geoplotlib.utils import DataAccessObject

test_data = DataAccessObject({
    'lat': np.array([51.5074, 48.8566, 52.5200]),
    'lon': np.array([-0.1278, 2.3522, 13.4050])
})

geoplotlib.dot(test_data)
geoplotlib.show()

In [16]:
print("Total cities:", len(dataset))

test_cities = dataset.head(1000)

geoplotlib.dot(geo_data(test_cities))
geoplotlib.show()

Total cities: 3173958
